# 04 · Live Demo Dashboard: Road Damage & Severity Inspector

**Computer Vision Group Project · IE School of Science & Technology**

An interactive **Streamlit** app for the presentation. Drop in a *new* road
image (a phone photo, a frame from the web, anything, it does **not** need to
be from the dataset) and the app returns the detected potholes/cracks, each
defect's severity, and an overall road-condition verdict.

On Colab, the app is served through a **Cloudflare tunnel** that prints a
**public URL** which can be opened on any device during the live demo.

In [14]:
# === Colab setup: mount Drive and locate the submitted project folder ===
import sys, os, glob, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not on Colab, using local paths.')

def find_project_dir():
    pats = ['/content/drive/MyDrive/**/CompVis Group Project',
            '/content/drive/Shareddrives/**/CompVis Group Project',
            '/content/drive/MyDrive/CompVis Group Project']
    for p in pats:
        hits = glob.glob(p, recursive=True)
        if hits:
            return hits[0]
    return os.path.abspath('..')  # local fallback (notebooks/ -> project root)

PROJECT_DIR = find_project_dir()
print('PROJECT_DIR =', PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DIR = /content/drive/MyDrive/CompVis Group Project


## 0. Reproducibility note: embedded project source

For reproducibility in Google Colab, this notebook writes its helper modules to `/content/src/` at runtime. This keeps the submitted notebook self-contained: the teacher only needs the project folder and the `Data/` directory included with this submission.

In [15]:
# Create the package directory and make it importable.
import os, sys
os.makedirs('/content/src', exist_ok=True)
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

In [16]:
%%writefile /content/src/config.py
"""
Central configuration for the Road Damage Detection project.

Everything that the notebooks and the other modules need to agree on lives here:
the class taxonomy, the colour palette used for plotting, the default training
hyper-parameters and the weights of the rule-based severity model. Keeping these
in one place means a single edit propagates to the EDA, training, inference and
demo code without any duplication.
"""
from __future__ import annotations

import glob
import os
from dataclasses import dataclass, field
from pathlib import Path

# --------------------------------------------------------------------------- #
# Class taxonomy
# --------------------------------------------------------------------------- #
# The four RDD2022 damage categories we keep. The order defines the integer
# class id used in the YOLO label files (D00 -> 0, D10 -> 1, D20 -> 2, D40 -> 3).
CLASS_IDS: dict[str, int] = {
    "D00": 0,  # longitudinal crack  (runs along the driving direction)
    "D10": 1,  # transverse crack    (runs across the road)
    "D20": 2,  # alligator crack     (interconnected / fatigue cracking)
    "D40": 3,  # pothole
}
ID_TO_CLASS: dict[int, str] = {v: k for k, v in CLASS_IDS.items()}

# Human-readable names used in plots, the report and the demo UI.
CLASS_NAMES: dict[str, str] = {
    "D00": "Longitudinal crack",
    "D10": "Transverse crack",
    "D20": "Alligator crack",
    "D40": "Pothole",
}
# Ordered list of short codes, indexed by class id. This is what goes into the
# `names:` field of the Ultralytics data.yaml.
YOLO_NAMES: list[str] = [ID_TO_CLASS[i] for i in range(len(ID_TO_CLASS))]

# Distinct, colour-blind friendly RGB colours per class (for matplotlib / cv2).
CLASS_COLORS: dict[str, tuple[int, int, int]] = {
    "D00": (31, 119, 180),   # blue
    "D10": (255, 127, 14),   # orange
    "D20": (148, 103, 189),  # purple
    "D40": (214, 39, 40),    # red
}

# Severity is drawn with a fixed traffic-light palette so it reads instantly.
SEVERITY_COLORS: dict[str, tuple[int, int, int]] = {
    "Low": (44, 160, 44),     # green
    "Medium": (255, 193, 7),  # amber
    "High": (214, 39, 40),    # red
}
SEVERITY_LEVELS: list[str] = ["Low", "Medium", "High"]

# --------------------------------------------------------------------------- #
# Severity model parameters
# --------------------------------------------------------------------------- #
# The severity score has no ground truth in RDD2022, so it is an engineered,
# fully transparent heuristic. Each class carries a base severity that reflects
# its structural meaning, and the geometry of the box (size / extent) scales it.
# These numbers are documented and justified in report/REPORT.md and can be
# tuned in one place.
SEVERITY_BASE: dict[str, float] = {
    "D00": 0.25,  # a single longitudinal crack is usually minor
    "D10": 0.30,  # transverse cracks slightly worse (water ingress, edges)
    "D20": 0.55,  # alligator cracking signals fatigue / sub-base failure
    "D40": 0.60,  # potholes are an immediate hazard
}
# Thresholds that turn the continuous 0-1 score into a Low/Medium/High label.
SEVERITY_THRESHOLDS: tuple[float, float] = (0.34, 0.67)

# --------------------------------------------------------------------------- #
# Training defaults (reported verbatim in the notebook and the report)
# --------------------------------------------------------------------------- #
@dataclass
class TrainConfig:
    model_scales: list[str] = field(default_factory=lambda: ["n", "s", "m"])
    epochs: int = 100
    patience: int = 20          # early-stopping patience
    imgsz: int = 512            # native resolution of the RDD China-Drone tiles
    batch: int = 16
    optimizer: str = "AdamW"
    lr0: float = 1e-3           # initial learning rate
    lrf: float = 1e-2           # final lr factor (cosine schedule -> lr0*lrf)
    cos_lr: bool = True         # cosine learning-rate scheduler
    weight_decay: float = 5e-4
    warmup_epochs: float = 3.0
    seed: int = 42
    amp: bool = True            # automatic mixed precision
    # Data augmentation (Ultralytics keys). Tuned for small, top-down road tiles:
    # heavy mosaic, modest colour jitter, no vertical flip (gravity matters less
    # from a drone, but we keep it off to stay faithful to capture geometry).
    hsv_h: float = 0.015
    hsv_s: float = 0.5
    hsv_v: float = 0.4
    fliplr: float = 0.5
    flipud: float = 0.0
    mosaic: float = 1.0
    scale: float = 0.5
    translate: float = 0.1
    close_mosaic: int = 10      # disable mosaic for the last N epochs


TRAIN = TrainConfig()

# --------------------------------------------------------------------------- #
# Data-split configuration
# --------------------------------------------------------------------------- #
SPLIT_RATIOS: tuple[float, float, float] = (0.70, 0.20, 0.10)  # train / val / test
SPLIT_SEED: int = 42


# --------------------------------------------------------------------------- #
# Path discovery
# --------------------------------------------------------------------------- #
def find_project_dir() -> Path:
    """Locate the project root both locally and inside Google Colab.

    On Colab the folder lives somewhere under a mounted Drive; we glob for it so
    the notebooks do not hard-code a brittle absolute path. Locally we fall back
    to the parent of this file's `src/` directory.
    """
    patterns = [
        "/content/drive/MyDrive/**/CompVis Group Project",
        "/content/drive/Shareddrives/**/CompVis Group Project",
        "/content/drive/MyDrive/CompVis Group Project",
    ]
    for pattern in patterns:
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return Path(hits[0])
    # Local fallback: src/ -> project root
    return Path(__file__).resolve().parent.parent


def get_paths(project_dir: str | os.PathLike | None = None) -> dict[str, Path]:
    """Return the canonical set of paths used across the project."""
    root = Path(project_dir) if project_dir else find_project_dir()
    return {
        "root": root,
        "raw_images": root / "Data" / "images",
        "raw_annotations": root / "Data" / "annotations" / "xmls",
        "yolo_dataset": root / "outputs" / "yolo_dataset",
        "data_yaml": root / "outputs" / "yolo_dataset" / "data.yaml",
        "models": root / "models",
        "outputs": root / "outputs",
        "figures": root / "figures",
        "report": root / "report",
    }


Overwriting /content/src/config.py


In [17]:
%%writefile /content/src/severity.py
"""
Rule-based severity estimation for detected road damage.

RDD2022 carries no severity ground truth, so severity here is an *engineered,
fully transparent* index rather than a learned label. The design goal is that
every number can be explained to a grader in one sentence:

    severity = clip( base[class]  +  geometry_term ,  0, 1 )

* `base[class]` encodes structural meaning (a pothole starts more severe than a
  hairline longitudinal crack, see config.SEVERITY_BASE).
* `geometry_term` grows with how much of the frame the damage occupies and, for
  linear cracks, with their length and thickness.

The continuous score is then bucketed into Low / Medium / High. An image-level
"road condition index" aggregates all detections so the demo can give a single
verdict per photo. Limitations (scale ambiguity, no depth) are discussed in the
report; this module deliberately keeps the logic simple and inspectable.
"""
from __future__ import annotations

from dataclasses import dataclass

from . import config


@dataclass
class Detection:
    """A model detection in pixel xyxy coordinates."""
    cls: str                 # class code, e.g. "D40"
    conf: float              # detector confidence 0-1
    xmin: float
    ymin: float
    xmax: float
    ymax: float

    @property
    def width(self) -> float:
        return max(0.0, self.xmax - self.xmin)

    @property
    def height(self) -> float:
        return max(0.0, self.ymax - self.ymin)

    @property
    def area(self) -> float:
        return self.width * self.height


def _geometry_term(det: Detection, img_w: int, img_h: int) -> float:
    """Class-aware geometric contribution to severity, in roughly [0, 0.45].

    * Potholes / alligator cracking are area-driven: a large blown-out patch is
      far worse than a small one, so we use the square-root of the area fraction
      (sub-linear, because even a modest pothole is already a hazard).
    * Longitudinal / transverse cracks are length-and-thickness driven: a long,
      wide crack matters more than a short hairline one. We combine the relative
      length (longer side / image diagonal) with relative thickness.
    """
    img_area = float(img_w * img_h)
    area_frac = det.area / img_area if img_area else 0.0

    if det.cls in ("D40", "D20"):  # pothole, alligator -> area driven
        return min(0.45, 0.9 * (area_frac ** 0.5))

    # Linear cracks (D00, D10): length + thickness driven.
    long_side = max(det.width, det.height)
    short_side = max(1.0, min(det.width, det.height))
    diag = (img_w ** 2 + img_h ** 2) ** 0.5
    length_term = long_side / diag                      # 0-1
    thickness_term = short_side / max(img_w, img_h)      # small
    return min(0.45, 0.35 * length_term + 0.6 * thickness_term)


def severity_score(det: Detection, img_w: int, img_h: int) -> float:
    """Continuous severity in [0, 1] for a single detection."""
    base = config.SEVERITY_BASE.get(det.cls, 0.3)
    score = base + _geometry_term(det, img_w, img_h)
    return float(max(0.0, min(1.0, score)))


def severity_label(score: float) -> str:
    """Bucket a continuous score into Low / Medium / High."""
    low_t, high_t = config.SEVERITY_THRESHOLDS
    if score < low_t:
        return "Low"
    if score < high_t:
        return "Medium"
    return "High"


def score_detection(det: Detection, img_w: int, img_h: int) -> dict:
    """Full severity record for one detection (used by inference + demo)."""
    s = severity_score(det, img_w, img_h)
    return {
        "class": det.cls,
        "class_name": config.CLASS_NAMES.get(det.cls, det.cls),
        "confidence": round(float(det.conf), 3),
        "severity_score": round(s, 3),
        "severity": severity_label(s),
        "bbox": [round(det.xmin, 1), round(det.ymin, 1),
                 round(det.xmax, 1), round(det.ymax, 1)],
        "area_fraction": round(det.area / float(img_w * img_h), 4),
    }


def image_condition_index(
    detections: list[Detection], img_w: int, img_h: int
) -> dict:
    """Aggregate per-detection severities into a single road-condition verdict.

    The index is dominated by the worst single defect (a road with one deep
    pothole is unsafe regardless of how many hairline cracks it has) but is
    nudged upward when many defects coexist. Concretely:

        index = max_severity  +  0.05 * (n_defects - 1)   (capped at 1.0)

    which is then bucketed with the same Low/Medium/High thresholds.
    """
    if not detections:
        return {"index": 0.0, "level": "None", "n_defects": 0, "worst": None}

    scored = [score_detection(d, img_w, img_h) for d in detections]
    worst = max(scored, key=lambda r: r["severity_score"])
    index = min(1.0, worst["severity_score"] + 0.05 * (len(scored) - 1))
    return {
        "index": round(index, 3),
        "level": severity_label(index),
        "n_defects": len(scored),
        "worst": worst,
        "detections": scored,
    }


Overwriting /content/src/severity.py


In [18]:
%%writefile /content/src/visualization.py
"""
Visualisation helpers: draw detections and severity on images, and a few EDA
plotting utilities shared by the notebooks.

All drawing works on RGB numpy arrays so the functions are equally usable from a
notebook, a script, or the Gradio demo. Nothing here depends on a trained model;
the functions take already-computed detections / severity records.
"""
from __future__ import annotations

from collections import Counter
from pathlib import Path

import cv2
import numpy as np

from . import config


# --------------------------------------------------------------------------- #
# Drawing boxes
# --------------------------------------------------------------------------- #
def _draw_label(img: np.ndarray, text: str, x: int, y: int,
                color: tuple[int, int, int]) -> None:
    """Draw a filled label chip with readable text above a box corner."""
    font, scale, thick = cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1
    (tw, th), base = cv2.getTextSize(text, font, scale, thick)
    y_top = max(0, y - th - base - 4)
    cv2.rectangle(img, (x, y_top), (x + tw + 6, y_top + th + base + 4), color, -1)
    cv2.putText(img, text, (x + 3, y_top + th + 2), font, scale,
                (255, 255, 255), thick, cv2.LINE_AA)


def draw_detections(
    image: np.ndarray,
    severity_records: list[dict],
    color_by: str = "severity",
) -> np.ndarray:
    """Return a copy of `image` with boxes + labels drawn.

    `severity_records` is the list produced by `severity.score_detection`.
    `color_by` is either "severity" (traffic-light) or "class".
    """
    canvas = image.copy()
    for rec in severity_records:
        x1, y1, x2, y2 = (int(v) for v in rec["bbox"])
        if color_by == "severity":
            color = config.SEVERITY_COLORS[rec["severity"]]
        else:
            color = config.CLASS_COLORS.get(rec["class"], (0, 255, 0))
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
        label = (f"{rec['class']} {rec['confidence']:.2f} "
                 f"| {rec['severity']} {rec['severity_score']:.2f}")
        _draw_label(canvas, label, x1, y1, color)
    return canvas


def severity_banner(image: np.ndarray, condition: dict) -> np.ndarray:
    """Stamp the overall road-condition verdict as a banner across the top."""
    canvas = image.copy()
    level = condition.get("level", "None")
    color = config.SEVERITY_COLORS.get(level, (90, 90, 90))
    text = (f"ROAD CONDITION: {level}  "
            f"(index {condition.get('index', 0):.2f}, "
            f"{condition.get('n_defects', 0)} defects)")
    h, w = canvas.shape[:2]
    cv2.rectangle(canvas, (0, 0), (w, 30), color, -1)
    cv2.putText(canvas, text, (8, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                (255, 255, 255), 2, cv2.LINE_AA)
    return canvas


# --------------------------------------------------------------------------- #
# EDA plots (matplotlib), imported lazily so cv2-only use stays light
# --------------------------------------------------------------------------- #
def plot_class_distribution(counter: Counter, ax=None, title: str = "Class distribution"):
    import matplotlib.pyplot as plt
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    codes = config.YOLO_NAMES
    counts = [counter.get(c, 0) for c in codes]
    colors = [tuple(v / 255 for v in config.CLASS_COLORS[c]) for c in codes]
    bars = ax.bar([config.CLASS_NAMES[c] for c in codes], counts, color=colors)
    ax.set_title(title)
    ax.set_ylabel("Number of boxes")
    ax.tick_params(axis="x", rotation=20)
    for b, c in zip(bars, counts):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), str(c),
                ha="center", va="bottom", fontsize=9)
    return ax


def plot_boxes_per_image(annotations, ax=None):
    import matplotlib.pyplot as plt
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    counts = [len(a.boxes) for a in annotations]
    ax.hist(counts, bins=range(0, max(counts) + 2), color="#4C78A8",
            edgecolor="white", align="left")
    ax.set_title("Damage instances per image")
    ax.set_xlabel("boxes per image")
    ax.set_ylabel("number of images")
    return ax


def plot_bbox_size_distribution(annotations, ax=None):
    """Scatter of relative box width vs height, reveals tiny/elongated cracks."""
    import matplotlib.pyplot as plt
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    for ann in annotations:
        for b in ann.boxes:
            color = tuple(v / 255 for v in config.CLASS_COLORS[b.cls])
            ax.scatter(b.width / ann.width, b.height / ann.height,
                       s=12, alpha=0.5, color=color)
    ax.set_xlabel("relative width")
    ax.set_ylabel("relative height")
    ax.set_title("Bounding-box shape distribution")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    return ax


def make_gallery(image_paths, annotations_by_name, n: int = 6, cols: int = 3,
                 figsize=(14, 9)):
    """Grid of sample images with their VOC boxes drawn, for the EDA notebook."""
    import matplotlib.pyplot as plt
    paths = list(image_paths)[:n]
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, p in zip(axes, paths):
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        ann = annotations_by_name.get(Path(p).name)
        if ann:
            for b in ann.boxes:
                color = config.CLASS_COLORS[b.cls]
                cv2.rectangle(img, (int(b.xmin), int(b.ymin)),
                              (int(b.xmax), int(b.ymax)), color, 2)
        ax.imshow(img)
        ax.set_title(Path(p).name, fontsize=8)
    fig.tight_layout()
    return fig


Overwriting /content/src/visualization.py


In [19]:
%%writefile /content/src/inference.py
"""
Inference wrapper that ties the trained YOLO detector to the severity model.

`RoadDamageDetector` loads an Ultralytics weight file once and exposes a single
`analyze()` call that returns, for any image, the raw detections, their severity
records and the aggregated road-condition index, plus an annotated RGB image.
Both the inference notebook and the Gradio demo use this class, so detection and
severity behaviour are identical everywhere.
"""
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np

from . import config, visualization
from .severity import Detection, image_condition_index


class RoadDamageDetector:
    def __init__(self, weights: str | Path, conf: float = 0.25, iou: float = 0.6):
        # Imported here so the module can be inspected without ultralytics installed.
        from ultralytics import YOLO

        self.model = YOLO(str(weights))
        self.conf = conf
        self.iou = iou
        # Map model's own class indices -> our canonical class codes.
        self._names = self.model.names

    # ------------------------------------------------------------------ #
    def _to_detections(self, result) -> list[Detection]:
        dets: list[Detection] = []
        if result.boxes is None:
            return dets
        xyxy = result.boxes.xyxy.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()
        clss = result.boxes.cls.cpu().numpy().astype(int)
        for (x1, y1, x2, y2), cf, ci in zip(xyxy, confs, clss):
            code = self._names.get(int(ci), config.ID_TO_CLASS.get(int(ci), str(ci)))
            dets.append(Detection(code, float(cf), float(x1), float(y1),
                                  float(x2), float(y2)))
        return dets

    # ------------------------------------------------------------------ #
    def analyze(self, image: str | Path | np.ndarray) -> dict:
        """Run detection + severity on one image.

        `image` may be a path or an RGB numpy array. Returns a dict with the
        annotated image (RGB), the per-detection severity records and the
        overall condition index.
        """
        if isinstance(image, (str, Path)):
            bgr = cv2.imread(str(image))
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        else:
            rgb = image if image.ndim == 3 else cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

        h, w = rgb.shape[:2]
        result = self.model.predict(rgb[:, :, ::-1], conf=self.conf, iou=self.iou,
                                    verbose=False)[0]
        dets = self._to_detections(result)
        condition = image_condition_index(dets, w, h)
        records = condition.get("detections", [])

        annotated = visualization.draw_detections(rgb, records, color_by="severity")
        annotated = visualization.severity_banner(annotated, condition)
        return {
            "image": rgb,
            "annotated": annotated,
            "condition": condition,
            "detections": records,
        }

    # ------------------------------------------------------------------ #
    def analyze_batch(self, image_paths) -> list[dict]:
        return [self.analyze(p) for p in image_paths]


def load_best_detector(models_dir: str | Path, prefer: str = "m",
                       conf: float = 0.25) -> RoadDamageDetector:
    """Find and load the best available trained weight under `models_dir`.

    Looks for `road_damage_yolo26{scale}/weights/best.pt`, preferring the
    requested scale and falling back to whatever exists.
    """
    models_dir = Path(models_dir)
    order = [prefer] + [s for s in ["m", "s", "n"] if s != prefer]
    for scale in order:
        cand = list(models_dir.glob(f"*{scale}*/weights/best.pt"))
        if cand:
            return RoadDamageDetector(cand[0], conf=conf)
    any_best = list(models_dir.glob("**/best.pt"))
    if any_best:
        return RoadDamageDetector(any_best[0], conf=conf)
    raise FileNotFoundError(f"No trained best.pt found under {models_dir}")


Overwriting /content/src/inference.py


In [ ]:
%%writefile /content/app_streamlit.py
"""
Streamlit live-demo dashboard for the presentation.

A standalone Streamlit entry script (run with `streamlit run`, not imported).
A grader drops in a *new* road image (a phone photo, a dashcam frame, anything,
it need not be from the dataset) and immediately sees the detected potholes /
cracks, each defect's severity, and an overall road-condition verdict.

The UI is a purpose-built dark "inspection console": a hero banner, a KPI
verdict card with a safe -> critical gauge, a colour-coded defects table and a
class/severity legend. All styling is driven by the same palette defined in
`src/config.py`, so the dashboard, the annotated image and the report stay
visually consistent.

The detector is loaded once via `st.cache_resource`; every widget change reruns
the script, so uploading an image or moving the confidence slider re-analyses
in real time (no explicit button needed).
"""
from __future__ import annotations

import os
import sys

# The helper package lives at /content/src (written by the notebook bootstrap).
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import numpy as np
import streamlit as st
from PIL import Image

from src import config
from src.inference import load_best_detector


# --------------------------------------------------------------------------- #
# Palette helpers (kept in sync with src/config.py so nothing is hard-coded)
# --------------------------------------------------------------------------- #
def _hex(rgb) -> str:
    return "#%02x%02x%02x" % tuple(int(c) for c in rgb)


SEVERITY_HEX = {lvl: _hex(rgb) for lvl, rgb in config.SEVERITY_COLORS.items()}
SEVERITY_HEX["None"] = "#64748b"
CLASS_HEX = {code: _hex(rgb) for code, rgb in config.CLASS_COLORS.items()}
SEVERITY_EMOJI = {"Low": "🟢", "Medium": "🟡", "High": "🔴", "None": "⚪"}


def _accent_vars(level: str) -> str:
    r, g, b = config.SEVERITY_COLORS.get(level, (100, 116, 139))
    return (f"--accent:{_hex((r, g, b))};"
            f"--accent-soft:rgba({r},{g},{b},0.16);"
            f"--accent-line:rgba({r},{g},{b},0.55);")


# --------------------------------------------------------------------------- #
# Global stylesheet — a compact, self-contained dark theme
# --------------------------------------------------------------------------- #
CSS = """
<style>
:root {
  --rd-bg: #0a0f1c; --rd-panel: #121a2e; --rd-panel-soft: #0f1728;
  --rd-border: rgba(148,163,184,0.14); --rd-text: #e8eefc;
  --rd-muted: #9fb0cc; --rd-amber: #f59e0b;
}
.stApp {
  background:
    radial-gradient(1200px 520px at 12% -12%, rgba(245,158,11,0.12), transparent 60%),
    radial-gradient(1000px 620px at 100% -5%, rgba(56,189,248,0.10), transparent 55%),
    var(--rd-bg);
}
#MainMenu, footer, [data-testid="stToolbar"], [data-testid="stDecoration"] { visibility: hidden; }
header[data-testid="stHeader"] { background: transparent; }
.block-container { max-width: 1180px; padding-top: 2.2rem; padding-bottom: 3rem; }

/* bordered containers become dark cards */
[data-testid="stVerticalBlockBorderWrapper"] {
  background: var(--rd-panel); border: 1px solid var(--rd-border) !important;
  border-radius: 18px; box-shadow: 0 20px 42px -32px rgba(0,0,0,0.9);
}
[data-testid="stFileUploaderDropzone"] {
  background: var(--rd-panel-soft); border: 1px dashed var(--rd-border); border-radius: 14px;
}

/* ---- hero ------------------------------------------------------------- */
.rd-hero {
  position: relative; overflow: hidden; border-radius: 22px; padding: 30px 34px 34px;
  background: linear-gradient(135deg, #0f1b30 0%, #12233f 55%, #191a2f 100%);
  border: 1px solid var(--rd-border); box-shadow: 0 26px 55px -30px rgba(0,0,0,0.85);
  margin-bottom: 22px;
}
.rd-hero::after {
  content: ""; position: absolute; left: 0; right: 0; bottom: 0; height: 6px;
  background-image: repeating-linear-gradient(90deg, rgba(245,158,11,0.85) 0 40px, transparent 40px 78px);
  opacity: 0.85;
}
.rd-eyebrow { letter-spacing: .24em; text-transform: uppercase; font-size: 12px; font-weight: 800; color: var(--rd-amber); }
.rd-hero h1 { margin: 10px 0 8px; font-size: 34px; font-weight: 800; letter-spacing: -0.02em; color: var(--rd-text); }
.rd-hero p { margin: 0; max-width: 730px; font-size: 15px; color: var(--rd-muted); }
.rd-hero b { color: #ffd38a; font-weight: 700; }
.rd-chips { display: flex; flex-wrap: wrap; gap: 10px; margin-top: 20px; }
.rd-chip { padding: 6px 13px; border-radius: 999px; font-size: 12.5px; font-weight: 600; color: #cdd8ef; background: rgba(148,163,184,0.10); border: 1px solid var(--rd-border); }
.rd-chip b { color: var(--rd-amber); }

/* ---- section titles --------------------------------------------------- */
.rd-section-title { display: flex; align-items: center; gap: 8px; margin: 2px 2px 8px; font-weight: 700; font-size: 15px; color: var(--rd-text); }
.rd-section-title .n { color: var(--rd-muted); font-weight: 500; font-size: 13px; }

/* ---- verdict card ----------------------------------------------------- */
.rd-verdict { border-radius: 18px; padding: 22px 24px; margin-top: 8px; background: linear-gradient(160deg, var(--accent-soft), rgba(255,255,255,0)); border: 1px solid var(--rd-border); border-left: 6px solid var(--accent); }
.rd-v-head { display: flex; align-items: center; gap: 15px; }
.rd-badge { width: 56px; height: 56px; border-radius: 15px; display: grid; place-items: center; font-size: 27px; background: var(--accent-soft); border: 1px solid var(--accent-line); }
.rd-v-sub { font-size: 12px; letter-spacing: .16em; text-transform: uppercase; color: var(--rd-muted); }
.rd-v-level { font-size: 30px; font-weight: 800; line-height: 1.05; color: var(--rd-text); }
.rd-kpis { display: flex; flex-wrap: wrap; gap: 14px; margin-top: 20px; }
.rd-kpi { flex: 1; min-width: 130px; padding: 14px 16px; border-radius: 14px; background: var(--rd-panel-soft); border: 1px solid var(--rd-border); }
.rd-kpi .val { font-size: 22px; font-weight: 800; color: var(--rd-text); }
.rd-kpi .cap { margin-top: 4px; font-size: 11.5px; letter-spacing: .09em; text-transform: uppercase; color: var(--rd-muted); }
.rd-gauge { position: relative; height: 14px; margin-top: 22px; border-radius: 999px; background: linear-gradient(90deg, #16a34a 0%, #f59e0b 55%, #dc2626 100%); }
.rd-gauge-marker { position: absolute; top: 50%; width: 20px; height: 20px; border-radius: 50%; background: #fff; border: 3px solid var(--accent); transform: translate(-50%, -50%); box-shadow: 0 2px 9px rgba(0,0,0,0.55); }
.rd-gauge-scale { display: flex; justify-content: space-between; margin-top: 9px; font-size: 11.5px; color: var(--rd-muted); }

/* ---- defects table ---------------------------------------------------- */
.rd-table-wrap { overflow-x: auto; }
.rd-dtable { width: 100%; border-collapse: separate; border-spacing: 0 8px; font-size: 14px; }
.rd-dtable th { padding: 0 14px 4px; text-align: left; font-size: 11.5px; font-weight: 600; letter-spacing: .1em; text-transform: uppercase; color: var(--rd-muted); }
.rd-dtable td { padding: 12px 14px; color: var(--rd-text); background: var(--rd-panel-soft); border-top: 1px solid var(--rd-border); border-bottom: 1px solid var(--rd-border); }
.rd-dtable td:first-child { border-left: 1px solid var(--rd-border); border-radius: 12px 0 0 12px; }
.rd-dtable td:last-child { border-right: 1px solid var(--rd-border); border-radius: 0 12px 12px 0; }
.rd-dot { display: inline-block; width: 10px; height: 10px; border-radius: 50%; margin-right: 9px; vertical-align: middle; }
.rd-pill { display: inline-block; padding: 4px 12px; border-radius: 999px; font-size: 12.5px; font-weight: 700; color: #fff; }
.rd-bar { display: inline-block; width: 88px; height: 8px; margin-right: 9px; border-radius: 999px; overflow: hidden; vertical-align: middle; background: rgba(148,163,184,0.20); }
.rd-bar > i { display: block; height: 100%; border-radius: 999px; background: #38bdf8; }
.rd-num { color: var(--rd-muted); font-variant-numeric: tabular-nums; }

/* ---- legend & empty states ------------------------------------------- */
.rd-legend { display: flex; flex-wrap: wrap; align-items: center; gap: 14px 20px; padding: 14px 18px; border-radius: 18px; background: var(--rd-panel); border: 1px solid var(--rd-border); margin-top: 8px; }
.rd-legend .lab { font-size: 12px; font-weight: 700; letter-spacing: .1em; text-transform: uppercase; color: var(--rd-muted); }
.rd-legend .grp { display: flex; flex-wrap: wrap; gap: 14px; }
.rd-legend .item { display: flex; align-items: center; gap: 7px; font-size: 13px; color: #cdd8ef; }
.rd-empty { padding: 32px; text-align: center; color: var(--rd-muted); border: 1px dashed var(--rd-border); border-radius: 16px; background: var(--rd-panel-soft); }
.rd-empty .big { font-size: 34px; margin-bottom: 8px; }
.rd-empty b { color: var(--rd-text); }
.rd-footer { padding: 22px 0 4px; text-align: center; font-size: 12.5px; color: var(--rd-muted); }
</style>
"""


# --------------------------------------------------------------------------- #
# HTML fragment builders (all left-aligned, no blank lines: Streamlit-safe)
# --------------------------------------------------------------------------- #
def _hero_html() -> str:
    return (
        "<div class='rd-hero'>"
        "<div class='rd-eyebrow'>Computer Vision · IE School of Science &amp; Technology</div>"
        "<h1>🛣️ Road Damage &amp; Severity Inspector</h1>"
        "<p>Drop in any road photo — a phone snapshot, a dashcam frame, an image from the web —"
        " and the model localises <b>potholes</b> and <b>cracks</b>, grades each defect's"
        " severity, and returns a single, at-a-glance road-condition verdict.</p>"
        "<div class='rd-chips'>"
        "<span class='rd-chip'><b>YOLO26</b> · fine-tuned</span>"
        "<span class='rd-chip'><b>RDD2022</b> · 4 damage classes</span>"
        "<span class='rd-chip'><b>Transparent</b> severity index</span>"
        "<span class='rd-chip'><b>Real-time</b> inference</span>"
        "</div></div>"
    )


def _legend_html() -> str:
    cls_items = "".join(
        f"<span class='item'><span class='rd-dot' style='background:{CLASS_HEX[c]}'></span>"
        f"{config.CLASS_NAMES[c]}</span>"
        for c in config.YOLO_NAMES
    )
    sev_items = "".join(
        f"<span class='item'><span class='rd-pill' style='background:{SEVERITY_HEX[s]}'>{s}</span></span>"
        for s in config.SEVERITY_LEVELS
    )
    return (
        "<div class='rd-legend'>"
        "<span class='lab'>Damage classes</span>"
        f"<span class='grp'>{cls_items}</span>"
        "<span class='lab' style='margin-left:8px'>Severity</span>"
        f"<span class='grp'>{sev_items}</span>"
        "</div>"
    )


def _footer_html() -> str:
    return ("<div class='rd-footer'>CompVis group project · YOLO26 detector + engineered "
            "severity model · demo powered by Streamlit</div>")


def _empty_box(big: str, msg: str) -> str:
    return f"<div class='rd-empty'><div class='big'>{big}</div>{msg}</div>"


def _verdict_html(cond: dict) -> str:
    n = int(cond.get("n_defects", 0) or 0)
    if n == 0:
        return _empty_box("✅", "No damage detected above the current confidence "
                                "threshold. Lower the slider to probe for faint cracks.")
    level = cond.get("level", "None")
    index = float(cond.get("index", 0.0) or 0.0)
    pct = max(0, min(100, int(round(index * 100))))
    worst = cond.get("worst") or {}
    worst_name = worst.get("class_name", "—")
    worst_sev = worst.get("severity", "—")
    emoji = SEVERITY_EMOJI.get(level, "⚪")
    plural = "s" if n != 1 else ""
    return (
        f"<div class='rd-verdict' style='{_accent_vars(level)}'>"
        "<div class='rd-v-head'>"
        f"<div class='rd-badge'>{emoji}</div>"
        "<div><div class='rd-v-sub'>Overall road condition</div>"
        f"<div class='rd-v-level'>{level}</div></div></div>"
        "<div class='rd-kpis'>"
        f"<div class='rd-kpi'><div class='val'>{index:.2f}</div><div class='cap'>Condition index</div></div>"
        f"<div class='rd-kpi'><div class='val'>{n}</div><div class='cap'>Defect{plural} found</div></div>"
        f"<div class='rd-kpi'><div class='val'>{worst_name}</div><div class='cap'>Most severe · {worst_sev}</div></div>"
        "</div>"
        f"<div class='rd-gauge'><div class='rd-gauge-marker' style='left:{pct}%'></div></div>"
        "<div class='rd-gauge-scale'><span>0.0 · safe</span><span>0.5</span><span>1.0 · critical</span></div>"
        "</div>"
    )


def _table_html(records: list) -> str:
    if not records:
        return "<div class='rd-empty' style='padding:22px'>No defects to list yet.</div>"
    records = sorted(records, key=lambda r: r.get("severity_score", 0), reverse=True)
    rows = []
    for r in records:
        cls_color = CLASS_HEX.get(r["class"], "#64748b")
        sev_color = SEVERITY_HEX.get(r["severity"], "#64748b")
        conf = float(r["confidence"])
        rows.append(
            "<tr>"
            f"<td><span class='rd-dot' style='background:{cls_color}'></span>{r['class_name']}"
            f" <span class='rd-num'>· {r['class']}</span></td>"
            f"<td><span class='rd-bar'><i style='width:{int(round(conf * 100))}%'></i></span>"
            f"<span class='rd-num'>{conf:.2f}</span></td>"
            f"<td><span class='rd-pill' style='background:{sev_color}'>{r['severity']}</span></td>"
            f"<td class='rd-num'>{float(r['severity_score']):.2f}</td>"
            f"<td class='rd-num'>{float(r['area_fraction']) * 100:.1f}%</td>"
            "</tr>"
        )
    return (
        "<div class='rd-table-wrap'><table class='rd-dtable'><thead><tr>"
        "<th>Damage type</th><th>Confidence</th><th>Severity</th><th>Score</th><th>Coverage</th>"
        f"</tr></thead><tbody>{''.join(rows)}</tbody></table></div>"
    )


# --------------------------------------------------------------------------- #
# Detector (loaded once, cached across reruns)
# --------------------------------------------------------------------------- #
@st.cache_resource(show_spinner="Loading YOLO26 detector…")
def get_detector():
    project_dir = os.environ.get("PROJECT_DIR")  # passed by the launcher cell
    paths = config.get_paths(project_dir)
    return load_best_detector(paths["models"], prefer="m", conf=0.25)


# --------------------------------------------------------------------------- #
# App
# --------------------------------------------------------------------------- #
def main() -> None:
    st.set_page_config(page_title="Road Damage & Severity Inspector",
                       page_icon="🛣️", layout="wide")
    st.markdown(CSS, unsafe_allow_html=True)
    st.markdown(_hero_html(), unsafe_allow_html=True)

    detector = get_detector()

    left, right = st.columns([5, 7], gap="large")
    with left:
        with st.container(border=True):
            st.markdown("<div class='rd-section-title'>📤 Upload road image"
                        "<span class='n'>· any phone photo or web frame</span></div>",
                        unsafe_allow_html=True)
            uploaded = st.file_uploader("Road image", type=["jpg", "jpeg", "png", "bmp", "webp"],
                                        label_visibility="collapsed")
            conf = st.slider("Confidence threshold", 0.05, 0.9, 0.25, 0.05)
            st.caption("Analysis runs automatically when you upload or move the slider.")

    result = None
    if uploaded is not None:
        image = np.array(Image.open(uploaded).convert("RGB"))
        detector.conf = float(conf)
        with st.spinner("Analyzing road…"):
            result = detector.analyze(image)

    with right:
        with st.container(border=True):
            st.markdown("<div class='rd-section-title'>🔍 Detections &amp; severity"
                        "<span class='n'>· boxes coloured by severity</span></div>",
                        unsafe_allow_html=True)
            if result is not None:
                st.image(result["annotated"], use_container_width=True)
            else:
                st.markdown(_empty_box("🖼️", "The annotated result will appear here "
                                             "once you upload an image."),
                            unsafe_allow_html=True)

    if result is not None:
        st.markdown(_verdict_html(result["condition"]), unsafe_allow_html=True)
    else:
        st.markdown(_empty_box("🛣️", "Upload a road image to get an instant condition verdict."),
                    unsafe_allow_html=True)

    st.markdown("<div class='rd-section-title' style='margin-top:14px'>📋 Detected defects"
                "<span class='n'>· ranked by severity</span></div>", unsafe_allow_html=True)
    st.markdown(_table_html(result["detections"] if result else []), unsafe_allow_html=True)

    st.markdown(_legend_html(), unsafe_allow_html=True)
    st.markdown(_footer_html(), unsafe_allow_html=True)


main()


In [21]:
%%writefile /content/src/__init__.py
"""Road Damage Detection, embedded source package (self-contained notebook).

Modules are written to /content/src by the notebook's bootstrap section."""
__version__ = "1.0.0"


Overwriting /content/src/__init__.py


In [22]:
# Import the freshly written package (clear any stale cached modules).
for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]
from src import config
print('Embedded src package ready, classes:', config.YOLO_NAMES)

Embedded src package ready, classes: ['D00', 'D10', 'D20', 'D40']


In [ ]:
# Install dependencies (quiet). Safe to re-run; skips what is already present.
%pip install -q 'ultralytics>=8.4.0' 'streamlit>=1.32' supervision seaborn 2>/dev/null
import ultralytics; ultralytics.checks()

In [ ]:
# Load the detector once for the inline fallback below. The Streamlit app
# (launched in the next section) loads its own cached copy in its own process.
from src import config, inference
paths = config.get_paths(PROJECT_DIR)
detector = inference.load_best_detector(paths['models'], prefer='m', conf=0.25)
print('Detector ready:', detector.model.names)

## 1. Launch the dashboard

The cell below starts the **Streamlit** app (`app_streamlit.py`) as a local
server on the Colab VM and opens a **Cloudflare quick tunnel** so it gets a
public `https://…trycloudflare.com` URL, no login or password page, that can be
opened on any device during the live demo. The first image analysed triggers a
one-off model warm-up, after which inference is near-instant.

The demo loads **YOLO26m**. Although YOLO26s is the best detector by mAP, we
deploy YOLO26m here because it has the highest recall (about 0.58). For a
maintenance triage tool, not missing damage (a false negative) matters more than
the occasional spurious box, which the operator can simply dismiss with the
confidence slider.

In [ ]:
# === Launch the Streamlit dashboard behind a public Cloudflare tunnel ===
import os, re, time, socket, subprocess, urllib.request
from IPython.display import HTML, display

PORT = 8501

# 1) Streamlit settings: headless, no telemetry, tunnel-friendly, dark theme.
os.makedirs('/root/.streamlit', exist_ok=True)
with open('/root/.streamlit/config.toml', 'w') as f:
    f.write(
        "[server]\nheadless = true\n" + f"port = {PORT}\n"
        "enableCORS = false\nenableXsrfProtection = false\n"
        "[browser]\ngatherUsageStats = false\n"
        "[theme]\nbase = \"dark\"\nprimaryColor = \"#f59e0b\"\n"
        "backgroundColor = \"#0a0f1c\"\nsecondaryBackgroundColor = \"#121a2e\"\n"
        "textColor = \"#e8eefc\"\n"
    )

# 2) Fetch the cloudflared binary once (no login, no password interstitial).
CF = '/content/cloudflared'
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CF)
    os.chmod(CF, 0o755)

# 3) Clean up any previous run so the cell is safe to re-run.
subprocess.run('pkill -f streamlit; pkill -f cloudflared', shell=True)
time.sleep(2)

# 4) Start Streamlit (pass PROJECT_DIR so the app skips the slow Drive glob).
env = {**os.environ, 'PROJECT_DIR': str(PROJECT_DIR)}
subprocess.Popen(
    f'streamlit run /content/app_streamlit.py --server.port {PORT} --server.headless true',
    shell=True, env=env,
    stdout=open('/content/streamlit.log', 'w'), stderr=subprocess.STDOUT)

# 5) Wait for the Streamlit port to accept connections.
for _ in range(60):
    with socket.socket() as s:
        if s.connect_ex(('127.0.0.1', PORT)) == 0:
            break
    time.sleep(1)

# 6) Open the Cloudflare tunnel and read the public URL from its log.
subprocess.Popen(
    f'{CF} tunnel --url http://localhost:{PORT} --no-autoupdate',
    shell=True, stdout=open('/content/cloudflared.log', 'w'), stderr=subprocess.STDOUT)

url = None
for _ in range(30):
    time.sleep(1)
    try:
        log = open('/content/cloudflared.log').read()
    except FileNotFoundError:
        continue
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', log)
    if m:
        url = m.group(0)
        break

if url:
    print('✅ Streamlit dashboard is live:', url)
    display(HTML(
        f"<a href='{url}' target='_blank' style='display:inline-block;margin-top:8px;"
        f"padding:12px 20px;border-radius:12px;background:linear-gradient(135deg,#f59e0b,#f97316);"
        f"color:#1b1206;font-weight:800;text-decoration:none;font-size:16px'>"
        f"🛣️ Open the Road Damage Inspector →</a>"))
else:
    print('⚠️ Tunnel URL not found yet, the server may still be starting.')
    print('   Re-run this cell, or inspect the logs below:')
    print('--- streamlit.log ---'); print(open('/content/streamlit.log').read()[-1500:])
    print('--- cloudflared.log ---'); print(open('/content/cloudflared.log').read()[-1500:])

## 2 · Fallback: analyse an uploaded image inline

If the share link is blocked on the venue network, this cell does the same
thing without the web server: upload a file and see the annotated result and
severity table directly in the notebook.

In [26]:
import matplotlib.pyplot as plt, pandas as pd
try:
    from google.colab import files
    up = files.upload()
    for fname in up:
        out = detector.analyze(fname)
        plt.figure(figsize=(9, 9)); plt.imshow(out['annotated']); plt.axis('off')
        plt.title(f"Road condition: {out['condition']['level']} (index {out['condition']['index']:.2f})")
        plt.show()
        if out['detections']:
            display(pd.DataFrame(out['detections'])[['class_name','confidence','severity','severity_score','area_fraction']])
        else:
            print('No damage detected above the confidence threshold.')
except Exception as e:
    print('Run this on Colab to upload an image. Details:', e)